# Hyderabad Traffic Analytics: bronze, silver, gold and congestion model

Run on **Azure Databricks (serverless)**. Replace `trafficdatavarma01` with your own storage account name.
Containers needed: `bronze`, `silver`, `gold` (see the main README for setup).

## 1. Read bronze
Bronze holds the raw CSV converted to Parquet by Azure Data Factory. Every column is still a string.

In [ ]:
bronze_path = "abfss://bronze@trafficdatavarma01.dfs.core.windows.net/traffic/"

df = spark.read.parquet(bronze_path)

print("Rows:", df.count())
df.printSchema()
display(df.limit(10))

## 2. Bronze to silver
Fix types, remove duplicates, add time features, fill missing speeds (junction + hour average), flag peak hours, save as Delta.

In [ ]:
from pyspark.sql import functions as F

bronze_path = "abfss://bronze@trafficdatavarma01.dfs.core.windows.net/traffic/"
silver_path = "abfss://silver@trafficdatavarma01.dfs.core.windows.net/traffic/"

df = spark.read.parquet(bronze_path)

# 1. Fix data types
clean = (df
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("is_weekend", F.col("is_weekend").cast("int"))
    .withColumn("vehicle_count", F.col("vehicle_count").cast("int"))
    .withColumn("avg_speed_kmph", F.col("avg_speed_kmph").cast("double"))
)

# 2. Remove duplicates and rows with no key fields
clean = clean.dropDuplicates(["timestamp", "junction"]).dropna(subset=["timestamp", "junction"])

# 3. Time features
clean = (clean
    .withColumn("date", F.to_date("timestamp"))
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("day_of_week", F.date_format("timestamp", "E"))
)

# 4. Fill missing speeds with the average for that junction and hour
avg_speed = (clean.groupBy("junction", "hour")
             .agg(F.round(F.avg("avg_speed_kmph"), 1).alias("avg_speed_fill")))

clean = (clean.join(avg_speed, ["junction", "hour"], "left")
         .withColumn("avg_speed_kmph", F.coalesce("avg_speed_kmph", "avg_speed_fill"))
         .drop("avg_speed_fill"))

# 5. Peak hour flag (weekday rush hours)
clean = clean.withColumn(
    "is_peak",
    F.when((F.col("is_weekend") == 0) &
           (F.col("hour").between(8, 11) | F.col("hour").between(17, 20)), 1).otherwise(0)
)

# 6. Write to silver
clean.write.mode("overwrite").format("delta").save(silver_path)

print("Silver rows:", clean.count())
print("Missing speeds left:", clean.filter(F.col("avg_speed_kmph").isNull()).count())
clean.printSchema()
display(clean.limit(10))

## 3. Silver to gold
Four aggregated tables for dashboards, saved as Delta and CSV.

In [ ]:
from pyspark.sql import functions as F

silver_path = "abfss://silver@trafficdatavarma01.dfs.core.windows.net/traffic/"
gold_base   = "abfss://gold@trafficdatavarma01.dfs.core.windows.net/"

silver = spark.read.format("delta").load(silver_path)

# Helper: 1 if congestion is High, else 0
silver = silver.withColumn("is_high", F.when(F.col("congestion_level") == "High", 1).otherwise(0))

# Gold 1: hourly pattern per junction (for peak-hour charts)
hourly = (silver.groupBy("junction", "hour", "is_weekend")
    .agg(F.round(F.avg("vehicle_count"), 1).alias("avg_vehicles"),
         F.round(F.avg("avg_speed_kmph"), 1).alias("avg_speed"),
         F.round(F.avg("is_high") * 100, 1).alias("pct_high_congestion"))
    .orderBy("junction", "hour"))

# Gold 2: daily summary per junction (for trend lines)
daily = (silver.groupBy("date", "day_of_week", "junction", "weather")
    .agg(F.sum("vehicle_count").alias("total_vehicles"),
         F.round(F.avg("avg_speed_kmph"), 1).alias("avg_speed"),
         F.sum("is_high").alias("high_congestion_intervals"))
    .orderBy("date", "junction"))

# Gold 3: junction hotspot ranking (for the map)
hotspots = (silver.groupBy("junction", "latitude", "longitude")
    .agg(F.round(F.avg("avg_speed_kmph"), 1).alias("avg_speed"),
         F.round(F.avg("is_high") * 100, 1).alias("pct_high_congestion"),
         F.sum("vehicle_count").alias("total_vehicles"))
    .orderBy(F.desc("pct_high_congestion")))

# Gold 4: weather impact
weather = (silver.groupBy("weather")
    .agg(F.round(F.avg("avg_speed_kmph"), 1).alias("avg_speed"),
         F.round(F.avg("is_high") * 100, 1).alias("pct_high_congestion"),
         F.round(F.avg("vehicle_count"), 1).alias("avg_vehicles")))

# Write each as Delta (for Databricks) and CSV (easy to load in Power BI)
tables = {"hourly_pattern": hourly, "daily_summary": daily,
          "junction_hotspots": hotspots, "weather_impact": weather}

for name, t in tables.items():
    t.write.mode("overwrite").format("delta").save(gold_base + "delta/" + name)
    t.coalesce(1).write.mode("overwrite").option("header", True).csv(gold_base + "csv/" + name)
    print(name, "->", t.count(), "rows")

display(hotspots)

## 4. Predict congestion one hour ahead
Random Forest with a time-based split (train 1-24 Oct, test 25-30 Oct) compared with a persistence baseline.
If `sklearn` is missing, run `%pip install scikit-learn` in a separate cell first.

In [ ]:
import pandas as pd
from pyspark.sql import functions as F, Window
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

silver_path = "abfss://silver@trafficdatavarma01.dfs.core.windows.net/traffic/"
gold_base   = "abfss://gold@trafficdatavarma01.dfs.core.windows.net/"

silver = spark.read.format("delta").load(silver_path)

# Target = congestion level 4 readings (1 hour) ahead; lag features = 15 min ago
w = Window.partitionBy("junction").orderBy("timestamp")
ml = (silver
    .withColumn("congestion_next_hour", F.lead("congestion_level", 4).over(w))
    .withColumn("speed_prev", F.lag("avg_speed_kmph", 1).over(w))
    .withColumn("vehicles_prev", F.lag("vehicle_count", 1).over(w))
    .dropna(subset=["congestion_next_hour", "speed_prev", "vehicles_prev"]))

pdf = ml.select("timestamp", "junction", "hour", "is_weekend", "is_peak", "weather",
                "vehicle_count", "avg_speed_kmph", "speed_prev", "vehicles_prev",
                "congestion_level", "congestion_next_hour").toPandas()
pdf["minute"] = pdf["timestamp"].dt.minute

# Features
X = pd.get_dummies(
    pdf[["junction", "weather", "hour", "minute", "is_weekend", "is_peak",
         "vehicle_count", "avg_speed_kmph", "speed_prev", "vehicles_prev"]],
    columns=["junction", "weather"])
y = pdf["congestion_next_hour"]

# Time-based split: train on Oct 1-24, test on Oct 25-30 (no shuffling)
cutoff = pd.Timestamp("2026-10-25")
train = pdf["timestamp"] < cutoff
X_train, X_test = X[train], X[~train]
y_train, y_test = y[train], y[~train]

model = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)

# Baseline: assume next hour = current level
baseline_acc = accuracy_score(y_test, pdf.loc[~train, "congestion_level"])
print("Baseline accuracy :", round(baseline_acc, 3))
print("Model accuracy    :", round(accuracy_score(y_test, pred), 3))
print()
print(classification_report(y_test, pred))

imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top features:")
print(imp.head(8).round(3))

# Save test predictions to gold for Power BI
results = pdf.loc[~train, ["timestamp", "junction", "weather", "congestion_level"]].copy()
results["actual_next_hour"] = y_test.values
results["predicted_next_hour"] = pred
res_sdf = spark.createDataFrame(results)
res_sdf.write.mode("overwrite").format("delta").save(gold_base + "delta/predictions")
res_sdf.coalesce(1).write.mode("overwrite").option("header", True).csv(gold_base + "csv/predictions")
print("Saved predictions:", res_sdf.count(), "rows")